### Walrus-оператор `:=` (Python 3.8+)

In [ ]:
print(x = 10) # error

In [ ]:
print(x := 10)

In [ ]:
def get_data() -> dict:
    return {"dttm": "2026-05-05T10:10:21", "name": "default_data"}

def process(data: dict) -> list:
    return [(k, v) for k, v in data.items()]


# Без walrus
data = get_data()
if data:
    process(data)

# С walrus - то же самое, но без дублирования вызова
if (data := get_data()):
    process(data)

# Фильтрация с побочным сохранением результата
def pow_int(value: int) -> int:
    return value ** 2

values = [1, 2, 3, 5, 6, 8, 11]
results = [y for x in values if (y := pow_int(x)) is not None]

print(results)

### Тернарный оператор

In [ ]:
class User:
    def __init__(self):
        self.is_active = True

user = User()

status = "активен" if user.is_active else "неактивен"
print(status)

In [ ]:
score = 99
grade = "A" if score >= 90 else "B" if score >= 75 else "C"
print(grade)

### Распаковка

In [ ]:
a, b, c = 1, 2, 3

first, *rest = [1, 2, 3, 4, 5]
print(f"first: {first}")
print(f"rest: {rest}\n")

*init, last = [1, 2, 3, 4, 5]
print(f"init: {init}")
print(f"last: {last}\n")

first, *middle, last = range(10)
print(f"first: {first}")
print(f"middle: {middle}")
print(f"last: {last}\n")


# Распаковка в вызовах функций
def total(a, b, c):
    return a + b + c

args = (1, 2, 3)
total(*args)

# Слияние словарей через **
defaults = {"debug": False, "timeout": 30}
config = {**defaults, "timeout": 60}   # переопределяем один ключ
print(f"defaults: {defaults}\n")
print(f"config: {config}\n")

### `*args` и `**kwargs`

In [ ]:
def log_call(*args, **kwargs):
    print(f"Позиционные: {args}, именованные: {kwargs}")

log_call(1, 2, name="Иван", role="analyst")
# Позиционные: (1, 2), именованные: {'name': 'Иван', 'role': 'analyst'}

### `zip` и `enumerate`

In [ ]:
names = ["Аня", "Борис", "Вера"]
scores = [85, 92, 78]

for i, (name, score) in enumerate(zip(names, scores), start=1):
    print(f"{i}. {name}: {score}")

# zip для транспонирования матрицы
matrix = [[1, 2, 3], [4, 5, 6]]
transposed = list(zip(*matrix))   # [(1, 4), (2, 5), (3, 6)]

### f-strings

In [ ]:
value = 1234567.891
print(f"{value=}")
print(f"{value:,.2f}")        # 1,234,567.89
print(f"{value:.2e}")         # 1.23e+06

name = "Мария"
print(f"{name=}")             # name='Мария'  - отладочный вывод, Python 3.8+

pi = 3.14159
print(f"{pi:>10.2f}")         # выравнивание по правому краю, ширина 10

### Декораторы

Декоратор - это функция, которая принимает функцию и возвращает новую функцию.

In [ ]:
import functools
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} выполнилась за {elapsed:.4f} сек")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(10_000_000)

In [46]:
@timer
def foo():
    pass

# эквивалентно
def foo():
    foo = timer(foo)

In [ ]:
import random

def maybe_random_error():
    if random.random() < 0.7:
        raise Exception("Случайная ошибка!")
    else:
        print("Успех!")

def retry(times=3): # default retry times
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Попытка {attempt + 1} не удалась: {e}")
            raise RuntimeError("Все попытки исчерпаны")
        return wrapper
    return decorator

@retry(times=5)
def fetch_from_api():
    maybe_random_error()

fetch_from_api()

### Базовые аннотации

In [ ]:
def greet(name: str, age: int = 18) -> str:
    return f"{name}, {age} лет"

numbers: list[int] = [1, 2, 3]
mapping: dict[str, float] = {"eur": 1.1, "usd": 1.0}

> В Python 3.9+ можно писать `list[int]`, `dict[str, int]` напрямую, без импорта `List`/`Dict` из `typing`.

### 2.2 `Optional`, `Union` и `|`

In [ ]:
from typing import Optional, Union

def find_user(user_id: int) -> Optional[dict]:
    # Optional[dict] == Union[dict, None]
    ...

def parse(value: Union[str, int]) -> float:
    ...

# Python 3.10+: короткий синтаксис
def parse_new(value: str | int) -> float:
    ...

def find_user_new(user_id: int) -> dict | None:
    ...

### 2.3 `Literal`

Ограничивает значение конкретным набором вариантов - полезно для флагов и режимов.

In [ ]:
from typing import Literal

def set_mode(mode: Literal["train", "eval", "predict"]) -> None:
    ...

set_mode("train")     # ок
set_mode("debug")     # mypy подсветит ошибку

### 2.4 `@dataclass` - замена "словарей с данными"

Частая ошибка в аналитическом коде - таскать данные в "голых" словарях (`{"name": ..., "value": ...}`), где легко опечататься в ключе и получить `KeyError` в рантайме. `dataclass` решает это через явную структуру.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Transaction:
    id: int
    category: str
    amount: float
    tags: list[str] = field(default_factory=list)

t = Transaction(id=1, category="еда", amount=450.0)
print(t)                     # автоматический __repr__
print(t.amount)               # обращение через атрибут, не через ["amount"]

`@dataclass` бесплатно генерирует `__init__`, `__repr__`, `__eq__`. Сравните со словарём:

In [ ]:
# Словарь: опечатка в ключе - ошибка только в рантайме
tx = {"categoty": "еда", "amount": 450.0}   # опечатка не будет замечена статически

# Dataclass: опечатка в поле - mypy/IDE подсветят сразу
tx = Transaction(categoty="еда", amount=450.0)   # ошибка обнаружена до запуска

### 2.5 Базовые `Protocol` и `Generic`

`Protocol` - "утиная типизация" со статической проверкой: объект подходит под тип, если у него есть нужные методы, без явного наследования.

In [ ]:
from typing import Protocol

class Reporter(Protocol):
    def report(self) -> str: ...

def print_report(obj: Reporter) -> None:
    print(obj.report())

# Любой класс с методом report() подходит - не нужно наследоваться от Reporter
class SalesSummary:
    def report(self) -> str:
        return "Отчёт по продажам"

print_report(SalesSummary())   # ок для mypy

`Generic` - типизированные контейнеры/классы, параметризуемые типом:

In [ ]:
from typing import Generic, TypeVar

T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

int_stack: Stack[int] = Stack()
int_stack.push(5)

### 2.6 Зачем `mypy` в команде

- Типы - это **исполняемая документация**: не нужно лезть в реализацию функции, чтобы понять, что она принимает и возвращает.
- Статический анализатор (`mypy`, `pyright`) ловит ошибки типов **до** запуска кода - особенно ценно в пайплайнах данных, где ошибка вылезает через час обработки.
- Упрощает ревью кода и рефакторинг больших кодовых баз.
- **Важно проговорить:** аннотации типов в Python - это подсказки, интерпретатор их не проверяет. Проверка - задача отдельного инструмента (`mypy`), который обычно встраивают в CI.

---

## Часть 3. Парадигмы программирования (45–70 мин)

Одну и ту же задачу можно решить разными "стилями мышления". Разберём три классических подхода на одном примере.

**Задача:** посчитать суммарные продажи по категориям из списка транзакций.

In [ ]:
transactions = [
    {"category": "еда", "amount": 450.0},
    {"category": "транспорт", "amount": 120.0},
    {"category": "еда", "amount": 300.0},
    {"category": "развлечения", "amount": 900.0},
    {"category": "транспорт", "amount": 80.0},
]

### 3.1 Процедурный стиль

In [ ]:
totals = {}
for tx in transactions:
    category = tx["category"]
    amount = tx["amount"]
    if category not in totals:
        totals[category] = 0.0
    totals[category] += amount

print(totals)

Прямолинейно, легко читается новичком, но: изменяемое состояние (`totals`) собирается вручную, легко забыть проверку `if category not in totals`.

### 3.2 Функциональный стиль

In [ ]:
from functools import reduce
from collections import defaultdict

# Вариант через reduce
def accumulate(acc: dict, tx: dict) -> dict:
    acc[tx["category"]] = acc.get(tx["category"], 0.0) + tx["amount"]
    return acc

totals = reduce(accumulate, transactions, {})

# Более "питонический" функциональный вариант - через defaultdict + comprehension
totals = defaultdict(float)
for category, amount in ((tx["category"], tx["amount"]) for tx in transactions):
    totals[category] += amount

Акцент на *чистых функциях* (без побочных эффектов) и неизменяемости данных на входе. `map`/`filter` пригодились бы, например, для предварительной фильтрации:

In [ ]:
big_transactions = list(filter(lambda tx: tx["amount"] > 200, transactions))
amounts = list(map(lambda tx: tx["amount"], transactions))

### 3.3 Объектно-ориентированный стиль

In [ ]:
from dataclasses import dataclass
from collections import defaultdict

@dataclass
class Transaction:
    category: str
    amount: float

class SalesAggregator:
    def __init__(self) -> None:
        self._totals: dict[str, float] = defaultdict(float)

    def add(self, transaction: Transaction) -> None:
        self._totals[transaction.category] += transaction.amount

    def as_dict(self) -> dict[str, float]:
        return dict(self._totals)

    def __repr__(self) -> str:
        return f"SalesAggregator({self.as_dict()})"

aggregator = SalesAggregator()
for tx in transactions:
    aggregator.add(Transaction(**tx))

print(aggregator)

Здесь состояние инкапсулировано внутри объекта, а не "болтается" в глобальной переменной. Легко расширить: добавить метод `top_category()`, `reset()`, логирование и т.д. без изменения внешнего интерфейса.

### 3.4 ООП: наследование vs композиция

In [ ]:
# Наследование - "является" (is-a)
class BaseAggregator:
    def __init__(self):
        self._totals = defaultdict(float)

    def add(self, category: str, amount: float) -> None:
        self._totals[category] += amount

class WeightedAggregator(BaseAggregator):
    def add(self, category: str, amount: float, weight: float = 1.0) -> None:
        self._totals[category] += amount * weight

# Композиция - "содержит" (has-a), обычно гибче
class ReportBuilder:
    def __init__(self, aggregator: BaseAggregator) -> None:
        self._aggregator = aggregator   # используем через композицию, не наследуемся

    def build(self) -> str:
        return "\n".join(f"{k}: {v:.2f}" for k, v in self._aggregator._totals.items())

**Практическое правило:** "composition over inheritance" - наследование создаёт жёсткую связь между классами и легко приводит к хрупким иерархиям при росте требований; композиция даёт гибкость и упрощает тестирование.

### 3.5 Dunder-методы

In [ ]:
class Money:
    def __init__(self, amount: float, currency: str = "RUB"):
        self.amount = amount
        self.currency = currency

    def __repr__(self) -> str:
        return f"Money({self.amount}, '{self.currency}')"

    def __eq__(self, other) -> bool:
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency

    def __add__(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise ValueError("Нельзя складывать разные валюты")
        return Money(self.amount + other.amount, self.currency)

a = Money(100)
b = Money(50)
print(a + b)          # Money(150, 'RUB')
print(a == Money(100))  # True

### 3.6 Метаклассы: как Python создаёт сами классы

До сих пор мы писали *классы*, которые создают *объекты*. Метаклассы отвечают на следующий вопрос: а кто создаёт сами классы? Это продвинутая тема, но она снимает "магию" с многих библиотек (Django ORM, Pydantic, ABC, dataclasses изнутри) и часто встречается на собеседованиях на позиции с Python.

**Ключевой факт:** в Python классы - это тоже объекты, и у них тоже есть тип.

In [ ]:
class SalesAggregator:
    pass

print(type(SalesAggregator))        # <class 'type'>
print(type(SalesAggregator()))      # <class '__main__.SalesAggregator'>

`type` - это метакласс по умолчанию: он "штампует" все классы, которые мы объявляем через `class`. Более того, `type` можно вызвать напрямую, без ключевого слова `class`:

In [ ]:
# Эквивалентные способы создать класс
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

Point = type("Point", (), {"__init__": lambda self, x, y: (setattr(self, "x", x), setattr(self, "y", y))})

**Свой метакласс** - это класс, унаследованный от `type`, который переопределяет, как создаются классы (не экземпляры классов, а сами классы):

In [ ]:
class LoggingMeta(type):
    def __new__(mcs, name, bases, namespace):
        print(f"Создаётся класс: {name}")
        return super().__new__(mcs, name, bases, namespace)

class Report(metaclass=LoggingMeta):
    pass
# При импорте модуля выведет: "Создаётся класс: Report"

**Зачем это нужно на практике** (а не просто "потому что можно"):

In [ ]:
# 1. Автоматическая регистрация подклассов - паттерн "плагины"
class PluginMeta(type):
    registry: dict[str, type] = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:  # не регистрируем базовый класс
            mcs.registry[name] = cls
        return cls

class BaseExporter(metaclass=PluginMeta):
    pass

class CSVExporter(BaseExporter):
    pass

class JSONExporter(BaseExporter):
    pass

print(PluginMeta.registry)   # {'CSVExporter': <class ...>, 'JSONExporter': <class ...>}


# 2. Валидация структуры класса на этапе объявления (в духе ORM/Pydantic)
class RequiresFieldsMeta(type):
    def __new__(mcs, name, bases, namespace):
        if bases and "validate" not in namespace:
            raise TypeError(f"{name} должен реализовать метод validate()")
        return super().__new__(mcs, name, bases, namespace)


# 3. Синглтон на уровне метакласса
class SingletonMeta(type):
    _instances: dict[type, object] = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class ConfigManager(metaclass=SingletonMeta):
    def __init__(self):
        self.settings = {}

a = ConfigManager()
b = ConfigManager()
print(a is b)   # True - один и тот же объект

**Важная оговорка для магистрантов:** метаклассы - мощный, но редко необходимый инструмент. Как говорил Тим Питерс: *"Metaclasses are deeper magic than 99% of users should ever worry about"*. Прежде чем писать свой метакласс, стоит проверить более простые альтернативы:

In [ ]:
# __init_subclass__ (Python 3.6+) - решает многие задачи без метакласса
class BasePlugin:
    registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        BasePlugin.registry[cls.__name__] = cls

class CSVPlugin(BasePlugin):
    pass

print(BasePlugin.registry)   # {'CSVPlugin': <class ...>} - без единого метакласса

| Задача | Метакласс нужен? | Альтернатива |
|---|---|---|
| Регистрация подклассов | Обычно нет | `__init_subclass__` |
| Валидация атрибутов класса при определении | Обычно нет | `__init_subclass__`, декоратор класса |
| Изменение поведения создания экземпляра (синглтон, кеш) | Иногда | `__new__` в самом классе, фабричная функция |
| ORM-подобное поведение полей (Django Model, Pydantic BaseModel) | Да, это классический сценарий | - |
| Библиотечный код, где нужен контроль над иерархией классов третьих сторон | Да | - |

**Практическое правило:** если сомневаетесь, нужен ли метакласс - скорее всего, не нужен. Метаклассы оправданы в первую очередь при написании библиотек/фреймворков, а не прикладного аналитического кода.

### 3.7 Обсуждение: что выбрать?

| Критерий | Процедурный | Функциональный | ООП |
|---|---|---|---|
| Порог входа | Низкий | Средний | Средний-высокий |
| Тестируемость | Средняя | Высокая (чистые функции) | Высокая (изоляция состояния) |
| Расширяемость при росте требований | Низкая | Средняя | Высокая |
| Типичное применение | Скрипты, разовые расчёты | Пайплайны трансформации данных | Долгоживущие системы, домены с состоянием |
| Риски | "Спагетти"-код при росте | Плохая читаемость при избыточной вложенности | Излишняя абстракция для простых задач |

**Ключевая мысль для студентов:** парадигма - это инструмент, а не религия. В одном модуле нормально сочетать функциональные трансформации данных (`pandas`, comprehensions) с объектной инкапсуляцией бизнес-логики.

---

## Часть 4. Исключения и контекстные менеджеры (70–85 мин)

### 4.1 `try / except / else / finally`

In [ ]:
def load_config(path: str) -> dict:
    try:
        with open(path) as f:
            data = f.read()
    except FileNotFoundError:
        print(f"Файл {path} не найден, используем значения по умолчанию")
        return {}
    except PermissionError as e:
        print(f"Нет доступа: {e}")
        raise
    else:
        # выполняется, только если исключений не было
        print("Файл успешно прочитан")
        return parse(data)
    finally:
        # выполняется всегда - используется для очистки ресурсов
        print("Попытка чтения конфигурации завершена")

### 4.2 Кастомные исключения

In [ ]:
class DataValidationError(Exception):
    """Базовое исключение для ошибок валидации данных."""

class MissingColumnError(DataValidationError):
    def __init__(self, column: str):
        self.column = column
        super().__init__(f"Отсутствует обязательная колонка: {column}")

def validate(df_columns: set[str]) -> None:
    required = {"category", "amount"}
    missing = required - df_columns
    if missing:
        raise MissingColumnError(missing.pop())

try:
    validate({"category"})
except MissingColumnError as e:
    print(f"Ошибка валидации: {e}, колонка: {e.column}")

Собственная иерархия исключений позволяет вызывающему коду ловить как конкретную ошибку, так и целую категорию (`except DataValidationError`).

### 4.3 `with` и протокол контекстного менеджера

In [ ]:
with open("data.csv") as f:
    content = f.read()
# файл гарантированно закрыт, даже если внутри блока было исключение

### 4.4 Собственный контекстный менеджер - через класс

In [ ]:
import time

class Timer:
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        elapsed = time.perf_counter() - self._start
        print(f"Блок выполнился за {elapsed:.4f} сек")
        return False   # False = не подавлять исключение, если оно было

with Timer():
    total = sum(range(10_000_000))

### 4.5 Собственный контекстный менеджер - через `contextlib.contextmanager`

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"Блок выполнился за {elapsed:.4f} сек")

with timer():
    total = sum(range(10_000_000))

**Сравнение подходов:** класс - явный и удобен, когда нужно хранить состояние между `__enter__`/`__exit__` или переиспользовать логику через наследование. `contextmanager`-декоратор - короче для простых одноразовых случаев.

---

## Часть 5. Итоговая демонстрация (в рамках занятия)

Разбираем на экране все три реализации задачи "суммарные продажи по категориям" (процедурная / функциональная / ООП, см. Часть 3) бок о бок и обсуждаем:

1. Что произойдёт с каждым подходом, если требования изменятся: нужно добавить фильтрацию по дате, поддержку нескольких валют, кеширование результата?
2. Какой вариант проще покрыть unit-тестами?
3. Какой вариант проще прочитать человеку, который видит код впервые?

---

## Введение в домашнее задание (85–90 мин)

**Постановка задачи:** реализовать модуль обработки транзакций, который:

- использует `@dataclass` для представления транзакции с типизированными полями (`typing`, `Literal` для категории или статуса);
- предоставляет функциональную и объектно-ориентированную реализацию агрегации (на выбор студента - какую делать основной, какую альтернативной);
- обрабатывает некорректные данные через кастomные исключения;
- использует хотя бы один собственный контекстный менеджер (например, для логирования времени обработки или безопасной записи результата в файл);
- проходит проверку `mypy` без ошибок.

Детальное описание, критерии оценки и шаблон репозитория - в файле задания в LMS.

---

## Материалы и ресурсы

- [Документация модуля `typing`](https://docs.python.org/3/library/typing.html)
- [PEP 526 - Syntax for Variable Annotations](https://peps.python.org/pep-0526/)
- [PEP 572 - Assignment Expressions (walrus)](https://peps.python.org/pep-0572/)
- [Документация `dataclasses`](https://docs.python.org/3/library/dataclasses.html)
- [Документация `functools`](https://docs.python.org/3/library/functools.html)
- [Документация `contextlib`](https://docs.python.org/3/library/contextlib.html)
- [mypy - официальная документация](https://mypy.readthedocs.io/)
- [Python Data Model - официальная документация (метаклассы, `__new__`, протокол создания класса)](https://docs.python.org/3/reference/datamodel.html#metaclasses)
- Рекомендуемое чтение: *Fluent Python* (Luciano Ramalho), главы 7–9 (функции как объекты, декораторы, замыкания), 11 (протоколы и ABC), 24 (метаклассы и объекты классов)
- Рекомендуемое чтение: *Python Distilled* (David Beazley), глава 7 (классы и ООП)